In [ ]:
!pip install kaggle

***O notebook segue um fluxo padrão de ML:***

**Configuração e imports:**

Define bibliotecas, tamanho das imagens (ex: 224x224) e parâmetros.

**Carregamento e pré-processamento:**

Usa ImageDataGenerator para normalizar e aumentar os dados.

**Construção do modelo:**
- Carrega o MobileNetV2 sem o topo
- Congela o modelo base
- Adiciona camadas finais para classificação

**Treinamento:**

Treina o modelo com dados de treino/validação.

**Avaliação / teste:**

Verifica desempenho com novas imagens.

In [ ]:
# =========================
# IMPORTS
# =========================
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import tensorflow_datasets as tfds

# =========================
# CONFIG
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5

# =========================
# CARREGAR DATASET (TFDS)
# =========================
print("📥 Carregando dataset via TFDS...")

(ds_train, ds_val), ds_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features['label'].num_classes

# =========================
# PREPROCESSAMENTO
# =========================
def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

train_ds = ds_train.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = ds_val.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# =========================
# DATA AUGMENTATION
# =========================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))

# =========================
# MODELO (TRANSFER LEARNING)
# =========================
print("🧠 Criando modelo...")

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================
# TREINAMENTO
# =========================
print("🚀 Treinando modelo...")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# =========================
# FINE-TUNING
# =========================
print("🔧 Fine-tuning...")

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, validation_data=val_ds, epochs=3)

# =========================
# SALVAR
# =========================
model.save("modelo_transfer_learning.keras")

print("💾 Modelo salvo com sucesso!")

📥 Carregando dataset via TFDS...
🧠 Criando modelo...


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,427,330 (9.26 MB)

 Trainable params: 166,786 (651.51 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

🚀 Treinando modelo...
Epoch 1/5
  6/582 ━━━━━━━━━━━━━━━━━━━━ 18:10 2s/step - accuracy: 0.5905 - loss: 0.8339

KeyboardInterrupt: 

In [ ]:
# =========================
# IMPORTS
# =========================
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np

# =========================
# CONFIG
# =========================
IMG_SIZE = (224, 224)

# =========================
# CARREGAR MODELO
# =========================
model = tf.keras.models.load_model("modelo_transfer_learning.keras")

# =========================
# FUNÇÃO DE PREDIÇÃO
# =========================
def predict_image(img_path):
    # Carregar imagem
    img = image.load_img(img_path, target_size=IMG_SIZE)

    # Converter para array
    img_array = image.img_to_array(img)

    # Normalizar (igual ao treino)
    img_array = img_array / 255.0

    # Adicionar dimensão do batch
    img_array = np.expand_dims(img_array, axis=0)

    # Fazer previsão
    prediction = model.predict(img_array)

    # Interpretar resultado
    classe = np.argmax(prediction)
    confianca = np.max(prediction)

    if classe == 0:
        resultado = "🐱 GATO"
    else:
        resultado = "🐶 CACHORRO"

    print(f"Resultado: {resultado}")
    print(f"Confiança: {confianca:.2%}")

# =========================
# TESTE COM IMAGEM
# =========================
predict_image("gato-bizarro-1.jpg")  # coloque o caminho da sua imagem aqui

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Resultado: 🐱 GATO
Confiança: 99.76%


Essa imagem serve como teste prático do modelo.

👉 O ponto aqui não é a imagem ser “normal”, mas sim:

ver se o modelo generaliza fora do dataset
testar comportamento em casos “estranhos”